# Data Quality Audit

**Project type:** Data inspection and cleaning  
**Tools:** Python, pandas

## Goal
Inspect a deliberately messy order dataset, quantify common data-quality issues, separate invalid records for review, and create a cleaned version for downstream analysis.

> **Dataset note:** `data/messy_orders.csv` is a small synthetic practice dataset. Some missing fields are deliberately retained in the cleaned output because there is not enough information to infer their correct values.

## 1. Load and inspect the data

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/messy_orders.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

orders_raw = pd.read_csv(DATA_PATH)
print(f"Rows: {len(orders_raw):,}")
print(f"Columns: {orders_raw.shape[1]}")
display(orders_raw.head())

Rows: 224
Columns: 6


,order_id,customer_id,order_date,product,quantity,unit_price
0,O0001,C023,2025-01-31,Pen,3,1.2
1,O0002,C053,2025-01-01,Pen,3,1.2
2,O0003,C005,2025-02-07,Lamp,8,16.0
3,O0004,C059,2025-02-06,Mouse,5,12.5
4,O0005,NaN,2025-02-03,Pen,4,1.2


## 2. Review column types and missing values

Missing percentages use the number of rows in the original file as the denominator.

In [2]:
orders_raw.info()

missing_summary = pd.DataFrame({
    "missing_count": orders_raw.isna().sum(),
    "missing_percent": (orders_raw.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

display(missing_summary)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     224 non-null    object 
 1   customer_id  216 non-null    object 
 2   order_date   224 non-null    object 
 3   product      221 non-null    object 
 4   quantity     224 non-null    int64  
 5   unit_price   224 non-null    float64
dtypes: float64(1), int64(1), object(4)
memory usage: 10.6+ KB


,missing_count,missing_percent
customer_id,8,3.57
product,3,1.34
order_id,0,0.00
order_date,0,0.00
quantity,0,0.00
unit_price,0,0.00


## 3. Find exact duplicate rows

`duplicated()` counts the extra copies after the first occurrence. The cleaning step keeps the first identical row and leaves the raw dataframe unchanged.

In [3]:
extra_duplicate_count = int(orders_raw.duplicated().sum())
print(f"Extra exact duplicate rows: {extra_duplicate_count}")

orders = orders_raw.drop_duplicates(keep="first").copy()
rows_after_deduplication = len(orders)
print(f"Rows after removing extra duplicates: {rows_after_deduplication}")

Extra exact duplicate rows: 4
Rows after removing extra duplicates: 220


## 4. Parse dates and set aside invalid dates

Invalid dates are saved for review before they are excluded from the analysis-ready table.

In [4]:
parsed_dates = pd.to_datetime(orders["order_date"], errors="coerce")
invalid_date_mask = parsed_dates.isna()

invalid_order_dates = orders.loc[invalid_date_mask].copy()
invalid_order_dates.to_csv(OUTPUT_DIR / "invalid_order_dates.csv", index=False)
print(f"Rows with invalid dates: {len(invalid_order_dates)}")

date_valid_orders = orders.loc[~invalid_date_mask].copy()
date_valid_orders["order_date"] = parsed_dates.loc[~invalid_date_mask]

Rows with invalid dates: 4


## 5. Check numeric values

For this exercise, each row is treated as a regular sale line: `quantity` must be positive and `unit_price` must be non-negative. In a real retail dataset, negative quantities may represent returns and should be reviewed rather than automatically removed.

In [5]:
invalid_quantity_mask = date_valid_orders["quantity"].le(0)
invalid_price_mask = date_valid_orders["unit_price"].lt(0)
invalid_value_mask = invalid_quantity_mask | invalid_price_mask

invalid_order_values = date_valid_orders.loc[invalid_value_mask].copy()
invalid_order_values.to_csv(OUTPUT_DIR / "invalid_order_values.csv", index=False)

print(f"Rows with quantity <= 0: {int(invalid_quantity_mask.sum())}")
print(f"Rows with unit_price < 0: {int(invalid_price_mask.sum())}")
print(f"Unique rows with at least one invalid quantity/price value: {len(invalid_order_values)}")

display(invalid_order_values.head())

orders_clean = date_valid_orders.loc[~invalid_value_mask].copy()

Rows with quantity <= 0: 3
Rows with unit_price < 0: 2
Unique rows with at least one invalid quantity/price value: 5


,order_id,customer_id,order_date,product,quantity,unit_price
25,O0026,C031,2025-03-22,Pen,-3,1.2
62,O0063,C046,2025-01-30,Lamp,5,-5.0
88,O0089,C027,2025-03-14,Mouse,-2,12.5
130,O0131,C029,2025-01-06,Lamp,5,-5.0
151,O0152,C039,2025-03-22,Lamp,-2,16.0


## 6. Review missing values after cleaning

No values are guessed or imputed. For example, rows with a missing customer ID remain in the dataset because the order information may still be useful for some analyses.

In [6]:
remaining_missing_summary = pd.DataFrame({
    "missing_count": orders_clean.isna().sum(),
    "missing_percent": (orders_clean.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

display(remaining_missing_summary)

,missing_count,missing_percent
customer_id,8,3.79
product,3,1.42
order_id,0,0.00
order_date,0,0.00
quantity,0,0.00
unit_price,0,0.00


## 7. Export the cleaned data and audit summary

In [7]:
orders_clean.to_csv(OUTPUT_DIR / "clean_orders.csv", index=False)
missing_summary.to_csv(OUTPUT_DIR / "missing_values_before_cleaning.csv")
remaining_missing_summary.to_csv(OUTPUT_DIR / "missing_values_after_cleaning.csv")

row_summary = pd.DataFrame({
    "stage": [
        "Original rows",
        "After removing extra exact duplicates",
        "After excluding invalid dates",
        "After excluding invalid quantity/price rows",
    ],
    "row_count": [
        len(orders_raw),
        rows_after_deduplication,
        len(date_valid_orders),
        len(orders_clean),
    ],
})
row_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)

display(row_summary)
print(f"Cleaned data and audit outputs saved in {OUTPUT_DIR}/")

,stage,row_count
0,Original rows,224
1,After removing extra exact duplicates,220
2,After excluding invalid dates,216
3,After excluding invalid quantity/price rows,211


Cleaned data and audit outputs saved in outputs/


## Interpretation and limitations

- Exact duplicates are treated as accidental copies; one copy is retained.
- Rows with unparseable dates and invalid quantity/price values are exported for review before exclusion.
- Missing values are reported rather than automatically filled.
- The cleaned dataset is suitable for practicing further analysis, but the cleaning rules should be reviewed before using a similar workflow with real business data.